# ARRAY-D-26-06140 — Revision experiments (Round 1)

This notebook re-runs every experiment the reviewers asked for under one corrected, leakage-free protocol.

**What is fixed compared with `Thesis.ipynb`**
1. **Endpoint.** In the Kaggle METABRIC export, `overall_survival = 1` means *alive*. The old notebook used it directly as the Cox event, so every METABRIC model learned the hazard of *being alive*, while TCGA used `1 = deceased`. Here the event is `1 = died` in both cohorts (checked by an assert against `death_from_cancer`).
2. **Feature selection** (100 genes, 40 mutations) is redone inside every training split (univariate Cox score test on training patients only). The old ranking used all 1,904 patients.
3. **Imputation/scaling** statistics come from training patients only.
4. **Inductive graphs.** Training patients are linked only to training patients; validation/test (and external) patients receive edges *from* their k nearest training patients and send no messages. The old graphs contained all 1,904 patients at once.
5. **Splits.** 10 stratified 70/15/15 splits (train/val/test); the validation set is used for early stopping and tuning only; all main numbers are on the untouched test sets. Bootstrap CIs (1,000) on the first split; corrected resampled t-tests across the 10 splits.
6. **Baselines on the same features.** CoxPH (penalty tuned on val), XGBoost with `survival:cox` (fixes the anti-concordant 0.345), DeepSurv, unified single graph, and a **no-graph** version of TripleGraph with an identical parameter count (same network, edges removed).
7. Subtype labels (PAM50, IntClust) are **not** model inputs; they are kept for adjustment and probing.

**Before running**
* Download `data_mutations.txt` for *TCGA-BRCA PanCancer Atlas 2018* from cBioPortal (study `brca_tcga_pan_can_atlas_2018` → Download) and put it in `DATA_DIR`. Without it, the cross-cohort mutation/TripleGraph analyses are skipped.
* Run once with `QUICK = True` (≈10 min) to catch errors, then with `QUICK = False`.
* Everything is written to `OUT_DIR`; the last cell zips it. **Share the zip** (it holds only CSV/PNG/JSON).

In [ ]:
!pip -q install lifelines scikit-survival xgboost torch_geometric

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CONFIG
import os, json, time, math, random, copy, warnings, itertools, shutil
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

DATA_DIR = "/content/drive/MyDrive/Thesis/Dataset"
OUT_DIR = "/content/drive/MyDrive/Thesis/Revision_R1_DSS"
METABRIC_CSV = f"{DATA_DIR}/METABRIC_RNA_Mutation.csv"
TCGA_CLIN = f"{DATA_DIR}/data_clinical_patient.txt"
TCGA_RNA = f"{DATA_DIR}/data_mrna_seq_v2_rsem_zscores_ref_all_samples.txt"
TCGA_MUT = f"{DATA_DIR}/data_mutations.txt"          # NEW file (see header)

QUICK = False                     # True = dry run (2 seeds, short training). Set False for the real run.
SEEDS = [42, 123, 456, 789, 2025, 7, 11, 21, 99, 314]
N_BOOT = 1000
N_RNA, N_MUT, MIN_MUT_COUNT = 100, 40, 10
K_DEFAULT = 10
METRICS_DEFAULT = {"clin": "euclidean", "rna": "cosine", "mut": "hamming", "all": "euclidean"}
ENDPOINT = "DSS"                  # "OS" (main) or "DSS" (secondary run)
EXCLUDE_MALE_TCGA = True
MAX_EPOCHS, PATIENCE = 300, 30
if QUICK:
    SEEDS, N_BOOT, MAX_EPOCHS, PATIENCE = SEEDS[:2], 100, 60, 10

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
json.dump(dict(QUICK=QUICK, SEEDS=SEEDS, N_BOOT=N_BOOT, N_RNA=N_RNA, N_MUT=N_MUT, K=K_DEFAULT,
               METRICS=METRICS_DEFAULT, ENDPOINT=ENDPOINT, MAX_EPOCHS=MAX_EPOCHS, PATIENCE=PATIENCE),
          open(f"{OUT_DIR}/config.json", "w"), indent=2)

def save(df, name):
    df.to_csv(f"{OUT_DIR}/{name}", index=False)
    print(f"saved {name}  ({len(df)} rows)")

In [ ]:
# CORE UTILITIES (numpy only)
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from scipy import stats


def harrell_c(time, risk, event):
    """Harrell's C. Comparable pairs: i had the event and t_i < t_j. Tied risk scores count 0.5.
    Higher risk = earlier event."""
    t = np.asarray(time, float); r = np.asarray(risk, float); e = np.asarray(event).astype(bool)
    ti, ri = t[e][:, None], r[e][:, None]
    comp = ti < t[None, :]
    n = comp.sum()
    if n == 0:
        return np.nan
    conc = ((ri > r[None, :]) & comp).sum()
    ties = ((ri == r[None, :]) & comp).sum()
    return (conc + 0.5 * ties) / n


def cox_score_test(X, time, event):
    """Univariate Cox score-test statistic (Breslow ties) for every column of X, vectorised.
    Used only to rank features on training patients."""
    X = np.asarray(X, float); t = np.asarray(time, float); e = np.asarray(event, float)
    o = np.argsort(t, kind="mergesort"); t, e, X = t[o], e[o], X[o]
    n = len(t)
    S0 = np.arange(n, 0, -1, dtype=float)
    S1 = np.cumsum(X[::-1], axis=0)[::-1]
    S2 = np.cumsum((X ** 2)[::-1], axis=0)[::-1]
    first = np.searchsorted(t, t, side="left")          # risk set = all with t_j >= t_i
    S0, S1, S2 = S0[first], S1[first], S2[first]
    ev = e == 1
    mean = S1[ev] / S0[ev][:, None]
    U = (X[ev] - mean).sum(0)
    I = (S2[ev] / S0[ev][:, None] - mean ** 2).sum(0)
    with np.errstate(divide="ignore", invalid="ignore"):
        s = np.where(I > 1e-12, U ** 2 / I, 0.0)
    return s


def make_split(event, seed, frac=(0.70, 0.15, 0.15)):
    idx = np.arange(len(event))
    tr, tmp = train_test_split(idx, test_size=1 - frac[0], random_state=seed, stratify=event)
    va, te = train_test_split(tmp, test_size=frac[2] / (frac[1] + frac[2]), random_state=seed,
                              stratify=event[tmp])
    return dict(tr=np.sort(tr), va=np.sort(va), te=np.sort(te))


def knn_edges_inductive(X, train_idx, k, metric):
    """Directed edge_index (2, E), messages flow source -> target.
    * train-train: symmetric kNN graph among training nodes only;
    * every non-training node receives edges from its k nearest TRAINING nodes and sends none.
    Hence training embeddings never depend on validation/test/external patients."""
    X = np.asarray(X, float); n = len(X)
    train_idx = np.asarray(train_idx)
    Xtr = X[train_idx]
    k_eff = min(k, len(train_idx) - 1)
    nn = NearestNeighbors(n_neighbors=k_eff + 1, metric=metric, algorithm="brute").fit(Xtr)
    _, ind = nn.kneighbors(Xtr)
    src, dst = [], []
    for a in range(len(train_idx)):
        row = [b for b in ind[a] if b != a][:k_eff]
        for b in row:
            src += [train_idx[b], train_idx[a]]; dst += [train_idx[a], train_idx[b]]
    other = np.setdiff1d(np.arange(n), train_idx)
    if len(other):
        _, ind_o = nn.kneighbors(X[other], n_neighbors=k_eff)
        src += list(train_idx[ind_o].ravel()); dst += list(np.repeat(other, k_eff))
    ei = np.unique(np.vstack([src, dst]), axis=1)
    return ei.astype(np.int64)


def shuffle_edges(ei, train_idx, rng):
    """Control graph: keeps every node's in-degree, replaces each source by a random training node."""
    ei = ei.copy()
    ei[0] = rng.choice(np.asarray(train_idx), size=ei.shape[1], replace=True)
    ei = ei[:, ei[0] != ei[1]]
    return ei


def graph_stats(ei, nodes, labels=None):
    """Statistics on the subgraph induced by `nodes` (typically training nodes)."""
    nodes = np.asarray(nodes); S = set(nodes.tolist())
    m = np.array([(a in S) and (b in S) for a, b in ei.T])
    sub = ei[:, m]
    und = {tuple(sorted(p)) for p in sub.T.tolist()}
    n = len(nodes); E = len(und)
    deg = pd.Series(np.concatenate([np.array([a for a, b in und]), np.array([b for a, b in und])])).value_counts()
    deg = deg.reindex(nodes, fill_value=0).values
    out = dict(n_nodes=n, n_undirected_edges=E, density=2 * E / (n * (n - 1)),
               deg_mean=deg.mean(), deg_sd=deg.std(), deg_min=deg.min(), deg_median=np.median(deg), deg_max=deg.max())
    for name, lab in (labels or {}).items():
        lab = pd.Series(lab)
        pairs = [(a, b) for a, b in und if pd.notna(lab.iloc[a]) and pd.notna(lab.iloc[b])]
        if not pairs:
            continue
        same = np.mean([lab.iloc[a] == lab.iloc[b] for a, b in pairs])
        p = lab.iloc[nodes].dropna().value_counts(normalize=True).values
        out[f"homophily_{name}"] = same
        out[f"expected_random_{name}"] = float((p ** 2).sum())
        out[f"cross_edges_{name}"] = 1 - same
    return out


def corrected_ttest(a, b, n_train, n_test):
    """Nadeau & Bengio (2003) corrected resampled t-test for paired scores over repeated splits."""
    d = np.asarray(a, float) - np.asarray(b, float)
    d = d[~np.isnan(d)]
    J = len(d)
    if J < 2:
        return dict(mean_diff=np.nan, t=np.nan, p=np.nan, J=J)
    v = d.var(ddof=1)
    se = np.sqrt((1 / J + n_test / n_train) * v)
    tval = d.mean() / se if se > 0 else np.nan
    p = 2 * stats.t.sf(abs(tval), J - 1) if se > 0 else np.nan
    return dict(mean_diff=d.mean(), sd_diff=d.std(ddof=1), t=tval, p=p, J=J)


def breslow_survival(r_tr, t_tr, e_tr, r_new, times):
    """S(t|x) = exp(-H0(t) * exp(r)), H0 by Breslow on the training risk scores."""
    r_tr = np.asarray(r_tr, float); c = r_tr.mean()
    w = np.exp(r_tr - c)
    t_tr = np.asarray(t_tr, float); e_tr = np.asarray(e_tr, float)
    ut = np.unique(t_tr[e_tr == 1])
    d = np.array([e_tr[t_tr == u].sum() for u in ut])
    den = np.array([w[t_tr >= u].sum() for u in ut])
    H = np.cumsum(d / den)
    H_at = np.array([H[ut <= s][-1] if np.any(ut <= s) else 0.0 for s in times])
    return np.exp(-np.outer(np.exp(np.asarray(r_new, float) - c), H_at))


def bootstrap_c(t, r, e, n_boot, seed=0):
    rng = np.random.default_rng(seed); n = len(t); vals = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        vals.append(harrell_c(t[i], r[i], e[i]))
    vals = np.array(vals)
    return np.nanpercentile(vals, 2.5), np.nanpercentile(vals, 97.5), vals


def paired_bootstrap_diff(t, r1, r2, e, n_boot, seed=0):
    rng = np.random.default_rng(seed); n = len(t); d = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        d.append(harrell_c(t[i], r1[i], e[i]) - harrell_c(t[i], r2[i], e[i]))
    d = np.array(d)
    return dict(diff=harrell_c(t, r1, e) - harrell_c(t, r2, e), lo=np.nanpercentile(d, 2.5),
                hi=np.nanpercentile(d, 97.5), p_two_sided=2 * min((d <= 0).mean(), (d >= 0).mean()))


def parse_stage(s):
    if pd.isna(s):
        return np.nan
    s = str(s).upper().replace("STAGE", "").strip()
    for roman, val in (("IV", 4.0), ("III", 3.0), ("II", 2.0), ("I", 1.0)):
        if s.startswith(roman):
            return val
    return np.nan


class Prep:
    """Imputation, scaling and feature selection, all estimated on training rows only."""

    def __init__(self, clin_cols, rna_pool, mut_pool, n_rna=N_RNA, n_mut=N_MUT, min_mut=MIN_MUT_COUNT):
        self.clin_cols, self.rna_pool, self.mut_pool = list(clin_cols), list(rna_pool), list(mut_pool or [])
        self.n_rna, self.n_mut, self.min_mut = n_rna, n_mut, min_mut

    def fit(self, D, tr, time, event):
        C = D.iloc[tr][self.clin_cols].astype(float)
        self.c_med = C.median(); C = C.fillna(self.c_med)
        self.c_mu, self.c_sd = C.mean(), C.std(ddof=0).replace(0, 1.0)
        R = D.iloc[tr][self.rna_pool].astype(float)
        self.r_med = R.median(); R = R.fillna(self.r_med)
        self.r_mu, self.r_sd = R.mean(), R.std(ddof=0).replace(0, 1.0)
        s = cox_score_test(((R - self.r_mu) / self.r_sd).values, time[tr], event[tr])
        self.rna_scores = pd.Series(s, index=self.rna_pool).sort_values(ascending=False)
        self.rna_sel = list(self.rna_scores.index[: self.n_rna])
        self.mut_sel = []
        if self.mut_pool:
            Mm = D.iloc[tr][self.mut_pool].fillna(0).astype(float)
            cand = [g for g in self.mut_pool if Mm[g].sum() >= self.min_mut]
            s = cox_score_test(Mm[cand].values, time[tr], event[tr])
            self.mut_scores = pd.Series(s, index=cand).sort_values(ascending=False)
            self.mut_sel = list(self.mut_scores.index[: self.n_mut])
        return self

    def transform(self, D, rna_stats="train"):
        out = {}
        C = D[self.clin_cols].astype(float).fillna(self.c_med)
        out["clin"] = ((C - self.c_mu) / self.c_sd).values.astype(np.float32)
        R = D[self.rna_sel].astype(float)
        if rna_stats == "train":
            R = R.fillna(self.r_med[self.rna_sel])
            out["rna"] = ((R - self.r_mu[self.rna_sel]) / self.r_sd[self.rna_sel]).values.astype(np.float32)
        else:   # label-free per-cohort standardisation (used for TCGA: different platform)
            R = R.fillna(R.median())
            out["rna"] = ((R - R.mean()) / R.std(ddof=0).replace(0, 1.0)).values.astype(np.float32)
        if self.mut_sel:
            out["mut"] = D[self.mut_sel].fillna(0).values.astype(np.float32)
        return out

In [ ]:
# SURVIVAL METRICS (lifelines / sksurv)
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index as ll_cindex
from sksurv.util import Surv
from sksurv.metrics import concordance_index_ipcw, cumulative_dynamic_auc, integrated_brier_score


def surv_struct(t, e):
    return Surv.from_arrays(event=np.asarray(e).astype(bool), time=np.asarray(t, float))


def evaluate(r_tr, t_tr, e_tr, r_ev, t_ev, e_ev, auc_times, ibs_max, tau, cens="train"):
    """Discrimination + calibration-type metrics on one evaluation set.
    cens='train': censoring distribution from the training set (internal);
    cens='self' : from the evaluation cohort itself (external cohort)."""
    t_ev = np.asarray(t_ev, float); e_ev = np.asarray(e_ev, int); r_ev = np.asarray(r_ev, float)
    res = dict(n=len(t_ev), events=int(e_ev.sum()), harrell_c=harrell_c(t_ev, r_ev, e_ev))
    y_ev = surv_struct(t_ev, e_ev)
    y_ref = surv_struct(t_tr, e_tr) if cens == "train" else y_ev
    try:
        res["uno_c"] = concordance_index_ipcw(y_ref, y_ev, r_ev, tau=tau)[0]
    except Exception as ex:
        res["uno_c"] = np.nan
    lo, hi = t_ev.min(), t_ev.max()
    for s in auc_times:
        try:
            res[f"td_auc_{s}m"] = float(cumulative_dynamic_auc(y_ref, y_ev, r_ev, [s])[0][0]) if lo < s < hi else np.nan
        except Exception:
            res[f"td_auc_{s}m"] = np.nan
    try:
        grid = np.linspace(max(3.0, lo + 1e-3), min(ibs_max, np.percentile(t_ev, 90)), 40)
        S = breslow_survival(r_tr, t_tr, e_tr, r_ev, grid)
        res["ibs"] = integrated_brier_score(y_ref, y_ev, S, grid)
    except Exception:
        res["ibs"] = np.nan
    try:
        df = pd.DataFrame(dict(lp=r_ev, T=t_ev, E=e_ev))
        res["calibration_slope"] = CoxPHFitter().fit(df, "T", "E").params_["lp"]
    except Exception:
        res["calibration_slope"] = np.nan
    return res


def km_groups(r_ev, t_ev, e_ev, cut):
    """High vs low risk using a cut-off fixed on the TRAINING risk distribution."""
    g = (np.asarray(r_ev) >= cut).astype(int)
    out = dict(cut=cut, n_high=int(g.sum()), n_low=int((1 - g).sum()))
    if g.sum() == 0 or g.sum() == len(g):
        return out, g
    lr = logrank_test(t_ev[g == 1], t_ev[g == 0], event_observed_A=e_ev[g == 1], event_observed_B=e_ev[g == 0])
    cph = CoxPHFitter().fit(pd.DataFrame(dict(high=g, T=t_ev, E=e_ev)), "T", "E")
    s = cph.summary.loc["high"]
    out.update(logrank_chi2=lr.test_statistic, logrank_p=lr.p_value, HR=s["exp(coef)"],
               HR_lo=s["exp(coef) lower 95%"], HR_hi=s["exp(coef) upper 95%"], HR_p=s["p"])
    return out, g


def decision_curve(risk_h, t, e, horizon, thresholds=np.round(np.arange(0.05, 0.55, 0.05), 2)):
    n = len(t); rows = []
    ev_all = 1 - KaplanMeierFitter().fit(t, e).predict(horizon)
    for pt in thresholds:
        hi = risk_h >= pt; nh = hi.sum()
        if nh == 0:
            nb = 0.0
        else:
            p_ev = 1 - KaplanMeierFitter().fit(t[hi], e[hi]).predict(horizon)
            nb = p_ev * nh / n - (1 - p_ev) * nh / n * pt / (1 - pt)
        rows.append(dict(threshold=pt, net_benefit_model=nb,
                         net_benefit_treat_all=ev_all - (1 - ev_all) * pt / (1 - pt), net_benefit_treat_none=0.0))
    return pd.DataFrame(rows)

In [ ]:
# MODELS (torch / PyG)
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from torch_geometric.nn import SAGEConv, GCNConv

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class Encoder(nn.Module):
    """Two graph layers. SAGEConv: h_v' = W_1 h_v + W_2 * AGG_{u in N(v)} h_u (root/self term included)."""

    def __init__(self, din, h, conv="sage", aggr="mean", dropout=0.3):
        super().__init__()
        if conv == "sage":
            self.c1, self.c2 = SAGEConv(din, h, aggr=aggr), SAGEConv(h, h, aggr=aggr)
        elif conv == "gcn":
            self.c1, self.c2 = GCNConv(din, h), GCNConv(h, h)
        self.dp = dropout

    def forward(self, x, ei):
        x = Fnn.dropout(Fnn.relu(self.c1(x, ei)), self.dp, self.training)
        return Fnn.relu(self.c2(x, ei))


class Fusion(nn.Module):
    def __init__(self, m, h, mode):
        super().__init__()
        self.m, self.mode = m, mode
        if m > 1 and mode == "concat":
            self.proj = nn.Linear(m * h, h)
        if m > 1 and mode == "attention":
            layer = nn.TransformerEncoderLayer(h, nhead=4, dim_feedforward=2 * h, dropout=0.1, batch_first=True)
            self.tr = nn.TransformerEncoder(layer, num_layers=2)

    def forward(self, zs):
        if self.m == 1:
            return zs[0]
        if self.mode == "concat":
            return Fnn.relu(self.proj(torch.cat(zs, 1)))
        if self.mode == "attention":
            return self.tr(torch.stack(zs, 1)).mean(1)
        return torch.stack(zs, 1).mean(1)


class Head(nn.Module):
    def __init__(self, h, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(h, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, z):
        return self.net(z).squeeze(-1)


class GraphSurv(nn.Module):
    def __init__(self, dims, h, conv, aggr, fusion, dropout):
        super().__init__()
        self.encs = nn.ModuleList([Encoder(d, h, conv, aggr, dropout) for d in dims])
        self.fuse = Fusion(len(dims), h, fusion)
        self.head = Head(h, dropout)

    def embed(self, xs, eis):
        return self.fuse([enc(x, ei) for enc, x, ei in zip(self.encs, xs, eis)])

    def forward(self, xs, eis):
        return self.head(self.embed(xs, eis))


class DeepSurvNet(nn.Module):
    def __init__(self, din, h, dropout):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(din, h), nn.ReLU(), nn.Dropout(dropout), nn.Linear(h, h), nn.ReLU())
        self.head = Head(h, dropout)

    def embed(self, xs, eis=None):
        return self.body(xs[0])

    def forward(self, xs, eis=None):
        return self.head(self.embed(xs))


def cox_loss(risk, time, event):
    """Negative Cox partial log-likelihood, Breslow ties: risk set R_i = {j : t_j >= t_i}."""
    o = torch.argsort(time, descending=True)
    r, e = risk[o], event[o]
    tn = time[o].detach().cpu().numpy()
    last = np.searchsorted(-tn, -tn, side="right") - 1
    lse = torch.logcumsumexp(r, 0)[torch.as_tensor(last, device=r.device)]
    return -((r - lse) * e).sum() / e.sum().clamp(min=1.0)


def n_params(m):
    return sum(p.numel() for p in m.parameters())

In [ ]:
# MODEL RUNNER
EMPTY = np.zeros((2, 0), dtype=np.int64)
DEFAULT_MODS = ["clin", "rna", "mut"]


def concat(F, mods):
    return np.hstack([F[m] for m in mods]).astype(np.float32)


def graph_inputs(F, spec, hp, train_idx, rng):
    """Node features and edge sets for one node set. Edges are inductive w.r.t. train_idx."""
    mods = [m for m in spec.get("mods", DEFAULT_MODS) if m in F]
    k = spec.get("k", hp.get("k", K_DEFAULT))
    metrics = {**METRICS_DEFAULT, **spec.get("metrics", {})}
    graph = spec.get("graph", "knn")
    if spec.get("unified"):
        Xs, names = [concat(F, mods)], ["all"]
    else:
        Xs, names = [F[m] for m in mods], mods
    eis = []
    for X, nm in zip(Xs, names):
        if graph == "none":
            ei = EMPTY
        else:
            ei = knn_edges_inductive(X, train_idx, k, metrics[nm])
            if graph == "shuffle":
                ei = shuffle_edges(ei, train_idx, rng)
        eis.append(ei)
    if spec.get("node_feat") == "shared" and not spec.get("unified"):
        Xs = [concat(F, mods)] * len(Xs)
    return Xs, eis


def to_t(Xs, eis):
    return ([torch.tensor(x, dtype=torch.float32, device=DEVICE) for x in Xs],
            [torch.tensor(e, dtype=torch.long, device=DEVICE) for e in eis])


def train_torch(model, xs, eis, t, e, tr, va, hp):
    tt = torch.tensor(t, dtype=torch.float32, device=DEVICE)
    et = torch.tensor(e, dtype=torch.float32, device=DEVICE)
    tr_t = torch.as_tensor(tr, device=DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=hp["lr"], weight_decay=hp["wd"])
    best, state, wait, best_ep = -1, None, 0, 0
    for ep in range(MAX_EPOCHS):
        model.train(); opt.zero_grad()
        loss = cox_loss(model(xs, eis)[tr_t], tt[tr_t], et[tr_t])
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0); opt.step()
        model.eval()
        with torch.no_grad():
            r = model(xs, eis).cpu().numpy()
        c = harrell_c(t[va], r[va], e[va])
        if c > best + 1e-4:
            best, state, wait, best_ep = c, copy.deepcopy(model.state_dict()), 0, ep
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(state)
    model.eval()
    return model, best, best_ep


def fit_cox(X, t, e, tr, va, grid=(0.01, 0.05, 0.1, 0.5, 1.0, 5.0)):
    cols = [f"f{i}" for i in range(X.shape[1])]
    df = pd.DataFrame(X, columns=cols)
    best = (-1, None, None)
    for p in grid:
        d = df.iloc[tr].copy(); d["T"] = t[tr]; d["E"] = e[tr]
        try:
            m = CoxPHFitter(penalizer=p).fit(d, "T", "E")
        except Exception:
            continue
        c = harrell_c(t[va], m.predict_log_partial_hazard(df.iloc[va]).values, e[va])
        if c > best[0]:
            best = (c, m, p)
    m = best[1]
    return (lambda Z: m.predict_log_partial_hazard(pd.DataFrame(Z, columns=cols)).values), best[0], best[2]


def fit_xgb(X, t, e, tr, va):
    import xgboost as xgb
    y = np.where(e == 1, np.maximum(t, 0.01), -np.maximum(t, 0.01))
    best = (-1, None, None)
    for depth in (2, 3, 4):
        m = xgb.XGBRegressor(objective="survival:cox", eval_metric="cox-nloglik", n_estimators=1000,
                             learning_rate=0.03, max_depth=depth, subsample=0.8, colsample_bytree=0.5,
                             min_child_weight=5, reg_lambda=1.0, early_stopping_rounds=50, random_state=0)
        m.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=False)
        c = harrell_c(t[va], m.predict(X[va], output_margin=True), e[va])
        if c > best[0]:
            best = (c, m, depth)
    m = best[1]
    return (lambda Z: m.predict(Z, output_margin=True)), best[0], best[2]


def run_model(spec, F, t, e, sp, hp, seed, externals=(), return_model=False):
    """Train on sp['tr'], select on sp['va']; returns risk for every node of F and every external set.
    externals: list of dict(name, F, mode in {'within','attach'})."""
    kind = spec["kind"]; mods = [m for m in spec.get("mods", DEFAULT_MODS) if m in F]
    tr, va = sp["tr"], sp["va"]
    rng = np.random.default_rng(seed)
    out = dict(ext={})
    set_seed(seed)
    if kind in ("cox", "xgb"):
        X = concat(F, mods)
        if spec.get("clin_subset") is not None:
            X = F["clin"][:, spec["clin_subset"]]
        pred, vc, tuned = (fit_cox if kind == "cox" else fit_xgb)(X, t, e, tr, va)
        out.update(risk=pred(X), val_c=vc, tuned=tuned, n_params=X.shape[1])
        for ex in externals:
            Xe = concat(ex["F"], mods) if spec.get("clin_subset") is None else ex["F"]["clin"][:, spec["clin_subset"]]
            out["ext"][ex["name"]] = pred(Xe)
        return out
    if kind == "mlp":
        X = concat(F, mods)
        model = DeepSurvNet(X.shape[1], hp["hidden"], hp["dropout"]).to(DEVICE)
        xs, eis = to_t([X], [EMPTY])
    else:
        Xs, E = graph_inputs(F, spec, hp, tr, rng)
        model = GraphSurv([x.shape[1] for x in Xs], hp["hidden"], spec.get("conv", "sage"),
                          spec.get("aggr", "mean"), spec.get("fusion", hp.get("fusion", "attention")),
                          hp["dropout"]).to(DEVICE)
        xs, eis = to_t(Xs, E)
    model, vc, ep = train_torch(model, xs, eis, t, e, tr, va, hp)
    with torch.no_grad():
        out.update(risk=model(xs, eis).cpu().numpy(), val_c=vc, tuned=ep, n_params=n_params(model))
        if return_model:
            out["Z"] = model.embed(xs, eis).cpu().numpy()
        for ex in externals:
            if kind == "mlp":
                xe, ee = to_t([concat(ex["F"], mods)], [EMPTY])
                out["ext"][ex["name"]] = model(xe, ee).cpu().numpy()
                continue
            if ex["mode"] == "within":
                n_ext = len(next(iter(ex["F"].values())))
                Xs_e, E_e = graph_inputs(ex["F"], spec, hp, np.arange(n_ext), rng)
                xe, ee = to_t(Xs_e, E_e)
                out["ext"][ex["name"]] = model(xe, ee).cpu().numpy()
            else:   # attach external patients to METABRIC training patients
                Fc = {m: np.vstack([F[m][tr], ex["F"][m]]) for m in ex["F"] if m in F}
                Xs_e, E_e = graph_inputs(Fc, spec, hp, np.arange(len(tr)), rng)
                xe, ee = to_t(Xs_e, E_e)
                out["ext"][ex["name"]] = model(xe, ee).cpu().numpy()[len(tr):]
    if return_model:
        out["model"] = model
    return out

In [ ]:
# LOAD COHORTS (corrected endpoint)
def read_cbio(path, **kw):
    """cBioPortal text files start with '#' metadata lines; skip exactly those."""
    n = 0
    with open(path) as f:
        for line in f:
            if line.startswith("#"):
                n += 1
            else:
                break
    return pd.read_csv(path, sep="\t", skiprows=n, low_memory=False, **kw)


META31 = ["patient_id", "age_at_diagnosis", "type_of_breast_surgery", "cancer_type", "cancer_type_detailed",
          "cellularity", "chemotherapy", "pam50_+_claudin-low_subtype", "cohort", "er_status_measured_by_ihc",
          "er_status", "neoplasm_histologic_grade", "her2_status_measured_by_snp6", "her2_status",
          "tumor_other_histologic_subtype", "hormone_therapy", "inferred_menopausal_state", "integrative_cluster",
          "primary_tumor_laterality", "lymph_nodes_examined_positive", "mutation_count",
          "nottingham_prognostic_index", "oncotree_code", "overall_survival_months", "overall_survival",
          "pr_status", "radio_therapy", "3-gene_classifier_subtype", "tumor_size", "tumor_stage",
          "death_from_cancer"]

met = pd.read_csv(METABRIC_CSV, low_memory=False)
ct = pd.crosstab(met["overall_survival"], met["death_from_cancer"].fillna("NA"))
print(ct)
assert ct.loc[1].drop(labels=["Living", "NA"], errors="ignore").sum() == 0, "overall_survival=1 should be 'Living' only"


def lowmap(s, mapping):
    v = s.astype(str).str.strip().str.lower().map(mapping)
    bad = s[v.isna() & s.notna()].unique()
    if len(bad):
        print("  unmapped values in", s.name, bad[:10])
    return v


MUT_COLS = [c for c in met.columns if c.endswith("_mut")]
RNA_COLS_MET = [c for c in met.columns if c not in META31 and c not in MUT_COLS]
assert all(pd.api.types.is_numeric_dtype(met[c]) for c in RNA_COLS_MET), "non-numeric column among genes"
print(f"METABRIC: {len(met)} patients, {len(RNA_COLS_MET)} mRNA genes, {len(MUT_COLS)} mutation genes")

clin = pd.DataFrame({
    "age": met["age_at_diagnosis"],
    "surgery_mastectomy": lowmap(met["type_of_breast_surgery"], {"mastectomy": 1, "breast conserving": 0}),
    "cellularity": lowmap(met["cellularity"], {"low": 0, "moderate": 1, "high": 2}),
    "chemotherapy": met["chemotherapy"], "hormone_therapy": met["hormone_therapy"], "radiotherapy": met["radio_therapy"],
    "er_pos": lowmap(met["er_status"], {"positive": 1, "negative": 0}),
    "pr_pos": lowmap(met["pr_status"], {"positive": 1, "negative": 0}),
    "her2_pos": lowmap(met["her2_status"], {"positive": 1, "negative": 0}),
    "postmenopausal": lowmap(met["inferred_menopausal_state"], {"post": 1, "pre": 0}),
    "grade": met["neoplasm_histologic_grade"], "lymph_nodes_pos": met["lymph_nodes_examined_positive"],
    "mutation_count": met["mutation_count"], "npi": met["nottingham_prognostic_index"],
    "tumor_size": met["tumor_size"], "stage": met["tumor_stage"]})
CLIN_S1 = list(clin.columns)                 # 16 clinical inputs (PAM50 / IntClust excluded)
CLIN_S2 = ["age", "stage"]                   # the only clinical variables shared with TCGA PanCan

rna_m = met[RNA_COLS_MET].copy(); rna_m.columns = ["g_" + c.lower() for c in RNA_COLS_MET]
mut_m = pd.DataFrame({"mut_" + c[:-4].upper(): ((~met[c].isna()) & (met[c].astype(str).str.strip() != "0")).astype(int)
                      for c in MUT_COLS})
pam50 = met["pam50_+_claudin-low_subtype"].replace({"NC": np.nan})
M = pd.concat([pd.DataFrame({
    "pid": met["patient_id"].astype(str),
    "os_time": met["overall_survival_months"].astype(float),
    "os_event": (met["overall_survival"] == 0).astype(int),                      # 1 = died (any cause)
    "dss_event": np.where(met["death_from_cancer"].isna(), np.nan,
                          (met["death_from_cancer"] == "Died of Disease").astype(float)),
    "pam50": pam50, "intclust": met["integrative_cluster"], "er_label": met["er_status"]}),
    clin, rna_m, mut_m], axis=1)
M["dss_time"] = M["os_time"]
if ENDPOINT == "DSS":
    M = M[M["dss_event"].notna()].reset_index(drop=True)
M["time"] = M[f"{ENDPOINT.lower()}_time"]; M["event"] = M[f"{ENDPOINT.lower()}_event"].astype(int)
PAM_LEVELS = ["LumA", "LumB", "Her2", "Basal", "Normal", "claudin-low"]
pam_dum = pd.get_dummies(M["pam50"]).reindex(columns=PAM_LEVELS, fill_value=0).astype(float).iloc[:, 1:]
RNA_POOL_S1 = list(rna_m.columns); MUT_POOL = list(mut_m.columns)
tM, eM = M["time"].values.astype(float), M["event"].values.astype(int)

# ---- TCGA
tc = read_cbio(TCGA_CLIN)
def col(df, *c):
    for x in c:
        if x in df.columns:
            return x
    raise KeyError(f"none of {c} in {list(df.columns)[:40]}")
c_stage = col(tc, "AJCC_PATHOLOGIC_TUMOR_STAGE", "PATH_STAGE", "AJCC_STAGE")
T = pd.DataFrame({
    "pid": tc[col(tc, "PATIENT_ID")].astype(str),
    "os_time": pd.to_numeric(tc[col(tc, "OS_MONTHS")], errors="coerce"),
    "os_event": tc[col(tc, "OS_STATUS")].astype(str).str.startswith("1").astype(int),
    "dss_time": pd.to_numeric(tc[col(tc, "DSS_MONTHS")], errors="coerce"),
    "dss_event": np.where(tc[col(tc, "DSS_STATUS")].isna(), np.nan,
                          tc[col(tc, "DSS_STATUS")].astype(str).str.startswith("1").astype(float)),
    "age": pd.to_numeric(tc[col(tc, "AGE", "DIAGNOSIS_AGE", "AGE_AT_DIAGNOSIS")], errors="coerce"),
    "stage": tc[c_stage].map(parse_stage),
    "sex": tc[col(tc, "SEX")],
    "pam50": tc[col(tc, "SUBTYPE")].astype(str).str.replace("BRCA_", "", regex=False).replace({"nan": np.nan})})
if EXCLUDE_MALE_TCGA:
    print("TCGA males excluded:", (T.sex.astype(str).str.lower() == "male").sum())
    T = T[T.sex.astype(str).str.lower() != "male"]

r = pd.read_csv(TCGA_RNA, sep="\t", low_memory=False)
r = r.dropna(subset=["Hugo_Symbol"]).drop_duplicates("Hugo_Symbol").set_index("Hugo_Symbol")
r = r.drop(columns=["Entrez_Gene_Id"], errors="ignore").T
r.columns = ["g_" + str(c).lower() for c in r.columns]
r = r.loc[:, ~r.columns.duplicated()]
r["pid"] = [s[:12] for s in r.index]; r["stype"] = [s[13:15] for s in r.index]
r = r[r.stype == "01"].drop_duplicates("pid").drop(columns="stype")
SHARED_GENES = [g for g in RNA_POOL_S1 if g in r.columns]
print("shared METABRIC/TCGA genes:", len(SHARED_GENES))
T = T.merge(r[["pid"] + SHARED_GENES], on="pid", how="inner")

HAS_TCGA_MUT = os.path.exists(TCGA_MUT)
if HAS_TCGA_MUT:
    maf = read_cbio(TCGA_MUT, usecols=["Hugo_Symbol", "Tumor_Sample_Barcode", "Variant_Classification"])
    maf["pid"] = maf.Tumor_Sample_Barcode.str[:12]
    profiled = set(maf.pid)
    NONSYN = {"Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del", "Frame_Shift_Ins", "In_Frame_Del",
              "In_Frame_Ins", "Splice_Site", "Nonstop_Mutation", "Translation_Start_Site"}
    ns = maf[maf.Variant_Classification.isin(NONSYN)]
    panel = [g[4:] for g in MUT_POOL]
    mt = pd.crosstab(ns.pid, ns.Hugo_Symbol).reindex(columns=panel, fill_value=0).clip(upper=1)
    mt.columns = ["mut_" + g for g in mt.columns]
    T = T.merge(mt, left_on="pid", right_index=True, how="left")
    T["mut_profiled"] = T.pid.isin(profiled)
    T[list(mt.columns)] = T[list(mt.columns)].fillna(0)
    print(f"TCGA mutation-profiled patients: {T.mut_profiled.sum()} / {len(T)}")
else:
    print("!! data_mutations.txt not found: cross-cohort mutation / TripleGraph analyses will be skipped")
    T["mut_profiled"] = False

T = T.dropna(subset=["os_time"]).reset_index(drop=True)
if ENDPOINT == "DSS":
    T = T[T["dss_event"].notna() & T["dss_time"].notna()].reset_index(drop=True)
T["time"] = T[f"{ENDPOINT.lower()}_time"]; T["event"] = T[f"{ENDPOINT.lower()}_event"].astype(int)
T_EVAL = T[T.mut_profiled].reset_index(drop=True) if HAS_TCGA_MUT else T
tT, eT = T_EVAL["time"].values.astype(float), T_EVAL["event"].values.astype(int)
print("TCGA evaluation cohort:", len(T_EVAL), "patients,", int(eT.sum()), "events")

In [ ]:
# COHORT SUMMARY (Reviewer 1 #7)
def summarise(name, D, endpoint, definition):
    t, e = D[f"{endpoint.lower()}_time"].values, D[f"{endpoint.lower()}_event"].values
    ok = ~np.isnan(t) & ~pd.isna(e); t, e = t[ok], e[ok].astype(int)
    fu = KaplanMeierFitter().fit(t, 1 - e).median_survival_time_      # reverse Kaplan-Meier
    km = KaplanMeierFitter().fit(t, e)
    return dict(cohort=name, endpoint=endpoint, definition=definition, time_unit="months", n=len(t),
                events=int(e.sum()), censored=int((1 - e).sum()), event_rate=e.mean(),
                median_followup_reverse_KM=fu, max_time=t.max(), surv_5y=float(km.predict(60)),
                surv_10y=float(km.predict(120)))

summ = pd.DataFrame([
    summarise("METABRIC", M, "OS", "death from any cause (overall_survival==0)"),
    summarise("METABRIC", M[M.dss_event.notna()], "DSS", "death_from_cancer=='Died of Disease'"),
    summarise("TCGA-BRCA", T, "OS", "OS_STATUS=='1:DECEASED'"),
    summarise("TCGA-BRCA", T[T.dss_event.notna()], "DSS", "DSS_STATUS starts with '1'")])
save(summ, "cohort_summary.csv"); display(summ)

if HAS_TCGA_MUT:   # mutation frequency comparison (Reviewer 1 #6)
    fm = pd.DataFrame({"gene": [g[4:] for g in MUT_POOL],
                       "freq_METABRIC": [M[g].mean() for g in MUT_POOL],
                       "freq_TCGA_profiled": [T_EVAL[g].mean() for g in MUT_POOL]})
    save(fm.sort_values("freq_METABRIC", ascending=False), "mutation_frequency_by_cohort.csv")

In [ ]:
# FEATURE SETTINGS
def features_S1(sp):
    prep = Prep(CLIN_S1, RNA_POOL_S1, MUT_POOL).fit(M, sp["tr"], tM, eM)
    F = prep.transform(M)
    F["pam50"] = pam_dum.values.astype(np.float32)
    return prep, F


def features_S2(sp, D=M, t=tM, e=eM, with_mut=HAS_TCGA_MUT):
    prep = Prep(CLIN_S2, SHARED_GENES, MUT_POOL if with_mut else []).fit(D, sp["tr"], t, e)
    return prep, prep.transform(D)


def ext_inputs(prep, D):
    return prep.transform(D, rna_stats="own")


AUC_M, IBS_M, TAU_M = (36, 60, 120), 120, 120
AUC_T, IBS_T, TAU_T = (24, 36, 60), 60, 60

dims = []
for seed in SEEDS:
    sp = make_split(eM, seed)
    p1, F1 = features_S1(sp)
    dims.append(dict(seed=seed, n_train=len(sp["tr"]), n_val=len(sp["va"]), n_test=len(sp["te"]),
                     events_train=int(eM[sp["tr"]].sum()), events_test=int(eM[sp["te"]].sum()),
                     clinical=F1["clin"].shape[1], rna=F1["rna"].shape[1], mutation=F1["mut"].shape[1],
                     total=F1["clin"].shape[1] + F1["rna"].shape[1] + F1["mut"].shape[1]))
    if seed == SEEDS[0]:
        pd.DataFrame({"rank": range(1, N_RNA + 1), "gene": [g[2:] for g in p1.rna_sel],
                      "score": p1.rna_scores.values[:N_RNA]}).to_csv(f"{OUT_DIR}/selected_genes_seed{seed}.csv", index=False)
        pd.DataFrame({"gene": [g[4:] for g in p1.mut_sel],
                      "n_mutated_train": [int(M.iloc[sp['tr']][g].sum()) for g in p1.mut_sel]}).to_csv(
            f"{OUT_DIR}/selected_mutations_seed{seed}.csv", index=False)
save(pd.DataFrame(dims), "feature_dimensions_and_splits.csv")

In [ ]:
# E0: hyper-parameter search (validation set, first split)
HP_GRID = [dict(lr=lr, hidden=h, dropout=d, wd=wd) for lr in (1e-3, 3e-4) for h in (32, 64)
           for d in (0.1, 0.3) for wd in (1e-4, 1e-3)]
if QUICK:
    HP_GRID = HP_GRID[:2]
SPEC_TUNE = {
    "DeepSurv": dict(kind="mlp"),
    "GraphSAGE-RNA": dict(kind="gnn", mods=["rna"]),
    "UnifiedGraph": dict(kind="gnn", unified=True),
    "TripleGraph": dict(kind="gnn", mods=["clin", "rna", "mut"]),
}
sp0 = make_split(eM, SEEDS[0]); _, F0 = features_S1(sp0)
rows, HP = [], {}
for name, spec in SPEC_TUNE.items():
    for hp in HP_GRID:
        o = run_model(spec, F0, tM, eM, sp0, hp, SEEDS[0])
        rows.append(dict(model=name, **hp, val_c=o["val_c"], best_epoch=o["tuned"]))
    best = max([r for r in rows if r["model"] == name], key=lambda r: r["val_c"])
    HP[name] = {k: best[k] for k in ("lr", "hidden", "dropout", "wd")}
    print(name, HP[name], round(best["val_c"], 4))
save(pd.DataFrame(rows), "e0_hparam_search.csv")
json.dump(HP, open(f"{OUT_DIR}/e0_selected_hparams.json", "w"), indent=2)
def hp_for(name):
    if name.startswith("GraphSAGE"):
        return HP["GraphSAGE-RNA"]
    if name.startswith("Unified"):
        return HP["UnifiedGraph"]
    if name.startswith("DeepSurv"):
        return HP["DeepSurv"]
    return HP["TripleGraph"]          # Dual/Triple variants, no-graph and shuffled controls share Triple's HPs

In [ ]:
# E1: main internal comparison (10 splits, same features)
NPI_IDX = [CLIN_S1.index("npi")]
E1_SPECS = {
    "CoxPH": dict(kind="cox"),
    "XGBoost-Cox": dict(kind="xgb"),
    "DeepSurv": dict(kind="mlp"),
    "GraphSAGE-RNA": dict(kind="gnn", mods=["rna"]),
    "UnifiedGraph": dict(kind="gnn", unified=True),
    "DualGraph": dict(kind="gnn", mods=["clin", "rna"]),
    "TripleGraph": dict(kind="gnn", mods=["clin", "rna", "mut"]),
    "TripleGraph-noGraph": dict(kind="gnn", mods=["clin", "rna", "mut"], graph="none"),
    "TripleGraph-shuffledGraph": dict(kind="gnn", mods=["clin", "rna", "mut"], graph="shuffle"),
    "NPI-Cox": dict(kind="cox", clin_subset=NPI_IDX),
    "Clinical+PAM50-Cox": dict(kind="cox", mods=["clin", "pam50"]),
}
runs, PRED0 = [], {}
for seed in SEEDS:
    sp = make_split(eM, seed); _, F = features_S1(sp)
    for name, spec in E1_SPECS.items():
        t0 = time.time()
        o = run_model(spec, F, tM, eM, sp, hp_for(name), seed)
        rsk = o["risk"]; tr, te = sp["tr"], sp["te"]
        ev = evaluate(rsk[tr], tM[tr], eM[tr], rsk[te], tM[te], eM[te], AUC_M, IBS_M, TAU_M)
        runs.append(dict(seed=seed, model=name, val_c=o["val_c"], n_params=o["n_params"], tuned=o["tuned"],
                         train_c=harrell_c(tM[tr], rsk[tr], eM[tr]), sec=time.time() - t0,
                         **{f"test_{k}": v for k, v in ev.items()}))
        if seed == SEEDS[0]:
            PRED0[name] = rsk
        print(f"seed {seed} {name:28s} val {o['val_c']:.4f} test {ev['harrell_c']:.4f}")
E1 = pd.DataFrame(runs); save(E1, "e1_internal_runs.csv")
sp0 = make_split(eM, SEEDS[0])
print("sanity: own Harrell vs lifelines on first split (TripleGraph):",
      harrell_c(tM[sp0['te']], PRED0['TripleGraph'][sp0['te']], eM[sp0['te']]),
      ll_cindex(tM[sp0['te']], -PRED0['TripleGraph'][sp0['te']], eM[sp0['te']]))
np.savez(f"{OUT_DIR}/e1_predictions_seed{SEEDS[0]}.npz", **PRED0, tr=sp0["tr"], va=sp0["va"], te=sp0["te"])

metric_cols = [c for c in E1.columns if c.startswith("test_") and c not in ("test_n", "test_events")]
agg = E1.groupby("model")[["val_c"] + metric_cols].agg(["mean", "std"])
agg.columns = [f"{a}_{b}" for a, b in agg.columns]
agg = agg.reset_index().sort_values("test_harrell_c_mean", ascending=False)
agg["n_params"] = agg.model.map(E1.groupby("model").n_params.first())
save(agg, "e1_summary_mean_sd.csv"); display(agg)

# bootstrap CI on the first split's test set
te = sp0["te"]; rows = []
for name, rsk in PRED0.items():
    lo, hi, _ = bootstrap_c(tM[te], rsk[te], eM[te], N_BOOT)
    rows.append(dict(model=name, test_c=harrell_c(tM[te], rsk[te], eM[te]), ci_lo=lo, ci_hi=hi))
save(pd.DataFrame(rows), f"e1_bootstrap_ci_seed{SEEDS[0]}.csv")

# paired comparisons: corrected resampled t-test over splits + paired bootstrap on first split
pairs = [("TripleGraph", "TripleGraph-noGraph"), ("TripleGraph", "TripleGraph-shuffledGraph"),
         ("TripleGraph", "UnifiedGraph"), ("TripleGraph", "GraphSAGE-RNA"), ("TripleGraph", "DualGraph"),
         ("TripleGraph", "CoxPH"), ("TripleGraph", "DeepSurv"), ("TripleGraph", "XGBoost-Cox"),
         ("TripleGraph", "Clinical+PAM50-Cox"), ("UnifiedGraph", "TripleGraph-noGraph")]
n_tr, n_te = len(sp0["tr"]), len(sp0["te"])
rows = []
for a, b in pairs:
    A = E1[E1.model == a].sort_values("seed").test_harrell_c.values
    B = E1[E1.model == b].sort_values("seed").test_harrell_c.values
    tt_ = corrected_ttest(A, B, n_tr, n_te)
    pb = paired_bootstrap_diff(tM[te], PRED0[a][te], PRED0[b][te], eM[te], N_BOOT)
    rows.append(dict(model_a=a, model_b=b, **{f"splits_{k}": v for k, v in tt_.items()},
                     **{f"boot_seed0_{k}": v for k, v in pb.items()}))
save(pd.DataFrame(rows), "e1_paired_tests.csv")

# KM on the held-out test set of the first split, cut-off = training median
rows = []
for name in ("TripleGraph", "CoxPH", "DeepSurv", "GraphSAGE-RNA"):
    rsk = PRED0[name]
    km, g = km_groups(rsk[te], tM[te], eM[te], np.median(rsk[sp0["tr"]]))
    rows.append(dict(model=name, cohort="METABRIC test", **km))
KM_ROWS = rows

In [ ]:
# E2: modality ablation, graph vs matched no-graph
SUBSETS = [["clin"], ["rna"], ["mut"], ["clin", "rna"], ["clin", "mut"], ["rna", "mut"], ["clin", "rna", "mut"]]
runs = []
for seed in SEEDS:
    sp = make_split(eM, seed); _, F = features_S1(sp)
    for mods in SUBSETS:
        for graph in ("knn", "none"):
            o = run_model(dict(kind="gnn", mods=mods, graph=graph), F, tM, eM, sp, HP["TripleGraph"], seed)
            te_ = sp["te"]
            runs.append(dict(seed=seed, modalities="+".join(mods), graph=graph, n_params=o["n_params"],
                             val_c=o["val_c"], test_c=harrell_c(tM[te_], o["risk"][te_], eM[te_])))
    print("E2 seed", seed, "done")
E2 = pd.DataFrame(runs); save(E2, "e2_ablation_runs.csv")
s = E2.groupby(["modalities", "graph"]).test_c.agg(["mean", "std"]).reset_index()
save(s, "e2_ablation_summary.csv"); display(s)
rows = []
for mods in SUBSETS:
    key = "+".join(mods)
    a = E2[(E2.modalities == key) & (E2.graph == "knn")].sort_values("seed").test_c.values
    b = E2[(E2.modalities == key) & (E2.graph == "none")].sort_values("seed").test_c.values
    rows.append(dict(modalities=key, comparison="graph - no graph", **corrected_ttest(a, b, n_tr, n_te)))
full = E2[(E2.modalities == "clin+rna+mut") & (E2.graph == "knn")].sort_values("seed").test_c.values
for mods in SUBSETS[:-1]:
    key = "+".join(mods)
    b = E2[(E2.modalities == key) & (E2.graph == "knn")].sort_values("seed").test_c.values
    rows.append(dict(modalities=key, comparison="all three - this subset (graph)", **corrected_ttest(full, b, n_tr, n_te)))
save(pd.DataFrame(rows), "e2_ablation_paired_tests.csv")

In [ ]:
# E3: sensitivity to k and distance metric
runs = []
K_LIST = (5, 10, 15, 20, 30)
METRIC_VARIANTS = {"default(euc/cos/ham)": {}, "all-euclidean": {"rna": "euclidean", "mut": "euclidean"},
                   "all-cosine": {"clin": "cosine", "mut": "cosine"}, "mut-jaccard": {"mut": "jaccard"}}
for seed in SEEDS:
    sp = make_split(eM, seed); _, F = features_S1(sp)
    for k in K_LIST:
        o = run_model(dict(kind="gnn", mods=DEFAULT_MODS, k=k), F, tM, eM, sp, HP["TripleGraph"], seed)
        runs.append(dict(seed=seed, factor="k", level=k, test_c=harrell_c(tM[sp["te"]], o["risk"][sp["te"]], eM[sp["te"]])))
    for nm, mv in METRIC_VARIANTS.items():
        o = run_model(dict(kind="gnn", mods=DEFAULT_MODS, metrics=mv), F, tM, eM, sp, HP["TripleGraph"], seed)
        runs.append(dict(seed=seed, factor="metric", level=nm, test_c=harrell_c(tM[sp["te"]], o["risk"][sp["te"]], eM[sp["te"]])))
    print("E3 seed", seed, "done")
E3 = pd.DataFrame(runs); save(E3, "e3_sensitivity_runs.csv")
save(E3.groupby(["factor", "level"]).test_c.agg(["mean", "std"]).reset_index(), "e3_sensitivity_summary.csv")

In [ ]:
# E4: aggregation / fusion with fixed inputs (R2 #1) + node features (R1 #8)
runs = []
E4_SPECS = {}
for conv, aggr in (("sage", "mean"), ("sage", "max"), ("gcn", "mean")):
    for fusion in ("concat", "attention", "mean"):
        E4_SPECS[f"clin+rna | {conv}-{aggr} | {fusion}"] = dict(kind="gnn", mods=["clin", "rna"], conv=conv, aggr=aggr, fusion=fusion)
E4_SPECS["clin+rna | no graph | attention"] = dict(kind="gnn", mods=["clin", "rna"], graph="none")
E4_SPECS["triple | restricted node features"] = dict(kind="gnn", mods=DEFAULT_MODS, node_feat="restricted")
E4_SPECS["triple | shared (concatenated) node features"] = dict(kind="gnn", mods=DEFAULT_MODS, node_feat="shared")
for seed in SEEDS:
    sp = make_split(eM, seed); _, F = features_S1(sp)
    for name, spec in E4_SPECS.items():
        o = run_model(spec, F, tM, eM, sp, HP["TripleGraph"], seed)
        runs.append(dict(seed=seed, variant=name, n_params=o["n_params"],
                         test_c=harrell_c(tM[sp["te"]], o["risk"][sp["te"]], eM[sp["te"]])))
    print("E4 seed", seed, "done")
E4 = pd.DataFrame(runs); save(E4, "e4_fusion_aggregation_runs.csv")
save(E4.groupby("variant").agg(test_c_mean=("test_c", "mean"), test_c_sd=("test_c", "std"),
                               n_params=("n_params", "first")).reset_index(), "e4_fusion_aggregation_summary.csv")

In [ ]:
# E5: graph diagnostics (R1 minor #4, #5)
sp = make_split(eM, SEEDS[0]); _, F = features_S1(sp)
labels = {"ER": M["er_label"].values, "PAM50": M["pam50"].values, "event": M["event"].values}
rows = []
for nm in ("clin", "rna", "mut"):
    ei = knn_edges_inductive(F[nm], sp["tr"], K_DEFAULT, METRICS_DEFAULT[nm])
    rows.append(dict(graph=nm, metric=METRICS_DEFAULT[nm], k=K_DEFAULT, **graph_stats(ei, sp["tr"], labels)))
ei = knn_edges_inductive(concat(F, DEFAULT_MODS), sp["tr"], K_DEFAULT, "euclidean")
rows.append(dict(graph="unified(clin+rna+mut)", metric="euclidean", k=K_DEFAULT, **graph_stats(ei, sp["tr"], labels)))
save(pd.DataFrame(rows), "e5_graph_statistics.csv"); display(pd.DataFrame(rows))

In [ ]:
# E6: out-of-fold analyses: clinical utility, subtype, embeddings
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

E6_SPECS = {k: E1_SPECS[k] for k in ("CoxPH", "DeepSurv", "TripleGraph", "TripleGraph-noGraph", "NPI-Cox", "Clinical+PAM50-Cox")}
OOF = {k: np.full(len(M), np.nan) for k in E6_SPECS}
OOF_S5 = {k: np.full(len(M), np.nan) for k in E6_SPECS}          # predicted 5-year survival
OOF_CUT = {k: np.full(len(M), np.nan) for k in E6_SPECS}         # per-fold training-median cut (z-scale)
FOLD = np.full(len(M), -1)
PROBE = {k: np.full(len(M), None, dtype=object) for k in ("TripleGraph", "TripleGraph-noGraph", "DeepSurv", "raw_inputs")}
skf = StratifiedKFold(5, shuffle=True, random_state=SEEDS[0])
for f, (trv, te_) in enumerate(skf.split(np.zeros(len(M)), eM)):
    tr_, va_ = train_test_split(trv, test_size=0.15, random_state=f, stratify=eM[trv])
    sp = dict(tr=np.sort(tr_), va=np.sort(va_), te=np.sort(te_)); FOLD[te_] = f
    _, F = features_S1(sp)
    lab = M["pam50"].values; fit_idx = np.concatenate([sp["tr"], sp["va"]])
    for name, spec in E6_SPECS.items():
        o = run_model(spec, F, tM, eM, sp, hp_for(name), SEEDS[0], return_model=(spec["kind"] in ("gnn", "mlp")))
        rsk = o["risk"]; mu, sd = rsk[sp["tr"]].mean(), rsk[sp["tr"]].std() + 1e-12
        OOF[name][te_] = (rsk[te_] - mu) / sd
        OOF_CUT[name][te_] = (np.median(rsk[sp["tr"]]) - mu) / sd
        OOF_S5[name][te_] = breslow_survival(rsk[sp["tr"]], tM[sp["tr"]], eM[sp["tr"]], rsk[te_], [60])[:, 0]
        if name in PROBE:
            Z = o["Z"]; ok = pd.notna(lab[fit_idx])
            sc = StandardScaler().fit(Z[fit_idx][ok])
            clf = LogisticRegression(max_iter=3000).fit(sc.transform(Z[fit_idx][ok]), lab[fit_idx][ok])
            PROBE[name][te_] = clf.predict(sc.transform(Z[te_]))
    Xraw = concat(F, DEFAULT_MODS); ok = pd.notna(lab[fit_idx])
    sc = StandardScaler().fit(Xraw[fit_idx][ok])
    clf = LogisticRegression(max_iter=3000).fit(sc.transform(Xraw[fit_idx][ok]), lab[fit_idx][ok])
    PROBE["raw_inputs"][te_] = clf.predict(sc.transform(Xraw[te_]))
    print("E6 fold", f, "done")
pd.DataFrame({"fold": FOLD, "time": tM, "event": eM, "pam50": M.pam50, "er": M.er_label,
              **{f"risk_{k}": v for k, v in OOF.items()}, **{f"S60_{k}": v for k, v in OOF_S5.items()}}).to_csv(
    f"{OUT_DIR}/e6_oof_predictions.csv", index=False)

rows = []
for name in E6_SPECS:
    g = (OOF[name] >= OOF_CUT[name]).astype(int)      # each fold's own training-median cut-off
    lr = logrank_test(tM[g == 1], tM[g == 0], event_observed_A=eM[g == 1], event_observed_B=eM[g == 0])
    s = CoxPHFitter().fit(pd.DataFrame(dict(high=g, T=tM, E=eM)), "T", "E").summary.loc["high"]
    rows.append(dict(model=name, oof_harrell_c=harrell_c(tM, OOF[name], eM), logrank_chi2=lr.test_statistic,
                     logrank_p=lr.p_value, HR_high_vs_low=s["exp(coef)"], HR_lo=s["exp(coef) lower 95%"],
                     HR_hi=s["exp(coef) upper 95%"]))
save(pd.DataFrame(rows), "e6_oof_discrimination_km.csv")

# calibration (5-year) by risk quintile + decision curves
cal, dca = [], []
for name in E6_SPECS:
    q = pd.qcut(OOF_S5[name], 5, labels=False, duplicates="drop")
    for b in np.unique(q):
        i = q == b
        cal.append(dict(model=name, quintile=int(b), n=int(i.sum()), predicted_S60=OOF_S5[name][i].mean(),
                        observed_KM_S60=float(KaplanMeierFitter().fit(tM[i], eM[i]).predict(60))))
    d = decision_curve(1 - OOF_S5[name], tM, eM, 60); d.insert(0, "model", name); dca.append(d)
save(pd.DataFrame(cal), "e6_calibration_5y.csv"); save(pd.concat(dca), "e6_decision_curve_5y.csv")

# does the risk score add information beyond subtype and clinical factors? (R1 #10, R2 #5)
base = pd.DataFrame({"age": M.age, "stage": M.stage.fillna(M.stage.median()), "grade": M.grade.fillna(M.grade.median()),
                     "nodes": M.lymph_nodes_pos, "size": M.tumor_size.fillna(M.tumor_size.median()),
                     "er": M.er_pos.fillna(0)}).join(pam_dum.add_prefix("pam_"))
base = (base - base.mean()) / base.std().replace(0, 1)
base["T"], base["E"] = tM, eM
m0 = CoxPHFitter(penalizer=1e-4).fit(base, "T", "E")
rows = []
for name in ("TripleGraph", "TripleGraph-noGraph", "DeepSurv", "CoxPH"):
    d = base.copy(); d["risk_z"] = OOF[name]
    m1 = CoxPHFitter(penalizer=1e-4).fit(d, "T", "E"); s = m1.summary.loc["risk_z"]
    lr = 2 * (m1.log_likelihood_ - m0.log_likelihood_)
    rows.append(dict(model=name, HR_per_SD_adjusted=s["exp(coef)"], lo=s["exp(coef) lower 95%"], hi=s["exp(coef) upper 95%"],
                     p=s["p"], LR_chi2_vs_clinical_PAM50=lr, LR_p=stats.chi2.sf(lr, 1),
                     c_clinical_PAM50=harrell_c(tM, m0.predict_log_partial_hazard(base).values, eM),
                     c_clinical_PAM50_plus_risk=harrell_c(tM, m1.predict_log_partial_hazard(d).values, eM)))
save(pd.DataFrame(rows), "e6_adjusted_for_subtype.csv")

rows = []
for grp_name, grp in [("ER", M.er_label), ("PAM50", M.pam50)]:
    for lvl in grp.dropna().unique():
        i = (grp == lvl).values
        if eM[i].sum() < 10:
            continue
        rows.append(dict(stratum=f"{grp_name}={lvl}", n=int(i.sum()), events=int(eM[i].sum()),
                         **{f"c_{k}": harrell_c(tM[i], OOF[k][i], eM[i]) for k in E6_SPECS}))
save(pd.DataFrame(rows), "e6_within_subgroup_cindex.csv")

lab = M["pam50"].values; ok = pd.notna(lab)
save(pd.DataFrame([dict(representation=k, balanced_acc=balanced_accuracy_score(lab[ok], v[ok].astype(str)),
                        macro_f1=f1_score(lab[ok], v[ok].astype(str), average="macro")) for k, v in PROBE.items()]),
     "e6_pam50_probe.csv")

In [ ]:
# E7: cross-cohort (harmonised features, R1 #1/#7, R2 #4)
E7_SPECS = {
    "CoxPH": dict(kind="cox"), "XGBoost-Cox": dict(kind="xgb"), "DeepSurv": dict(kind="mlp"),
    "GraphSAGE-RNA": dict(kind="gnn", mods=["rna"]), "UnifiedGraph": dict(kind="gnn", unified=True),
    "DualGraph": dict(kind="gnn", mods=["clin", "rna"]),
    "TripleGraph": dict(kind="gnn", mods=["clin", "rna", "mut"]),
    "TripleGraph-noGraph": dict(kind="gnn", mods=["clin", "rna", "mut"], graph="none"),
}
if not HAS_TCGA_MUT:
    E7_SPECS = {k: v for k, v in E7_SPECS.items() if not k.startswith("Triple")}
runs, PRED_T0 = [], {}
for seed in SEEDS:
    spM = make_split(eM, seed)
    prepM, FM = features_S2(spM)
    FT = ext_inputs(prepM, T_EVAL)
    spT = make_split(eT, seed)
    prepT, FTT = features_S2(spT, D=T_EVAL, t=tT, e=eT)          # TCGA-trained comparison (same features type)
    for name, spec in E7_SPECS.items():
        exts = [dict(name="TCGA_within", F=FT, mode="within"), dict(name="TCGA_attach", F=FT, mode="attach")]
        o = run_model(spec, FM, tM, eM, spM, hp_for(name), seed, externals=exts)
        rM = o["risk"]; trM = spM["tr"]
        base_row = dict(seed=seed, model=name)
        ev = evaluate(rM[trM], tM[trM], eM[trM], rM[spM["te"]], tM[spM["te"]], eM[spM["te"]], AUC_M, IBS_M, TAU_M)
        runs.append(dict(**base_row, trained_on="METABRIC", evaluated_on="METABRIC test", graph_mode="-", **ev))
        for mode in ("within", "attach"):
            if spec["kind"] not in ("gnn",) and mode == "attach":
                continue
            rT = o["ext"][f"TCGA_{mode}"]
            for subset, idx in (("TCGA all", np.arange(len(tT))), ("TCGA test", spT["te"])):
                ev = evaluate(rM[trM], tM[trM], eM[trM], rT[idx], tT[idx], eT[idx], AUC_T, IBS_T, TAU_T, cens="self")
                runs.append(dict(**base_row, trained_on="METABRIC", evaluated_on=subset, graph_mode=mode, **ev))
            if seed == SEEDS[0] and mode == "within":
                PRED_T0[name] = (rT, np.median(rM[trM]))
        o2 = run_model(spec, FTT, tT, eT, spT, hp_for(name), seed)
        r2 = o2["risk"]
        ev = evaluate(r2[spT["tr"]], tT[spT["tr"]], eT[spT["tr"]], r2[spT["te"]], tT[spT["te"]], eT[spT["te"]],
                      AUC_T, IBS_T, TAU_T)
        runs.append(dict(**base_row, trained_on="TCGA", evaluated_on="TCGA test", graph_mode="within", **ev))
        print(f"E7 seed {seed} {name} done")
E7 = pd.DataFrame(runs); save(E7, "e7_cross_cohort_runs.csv")
mc = ["harrell_c", "uno_c"] + [c for c in E7.columns if c.startswith("td_auc")] + ["ibs", "calibration_slope"]
s = E7.groupby(["model", "trained_on", "evaluated_on", "graph_mode"])[mc].agg(["mean", "std"])
s.columns = [f"{a}_{b}" for a, b in s.columns]; s = s.reset_index()
save(s, "e7_cross_cohort_summary.csv"); display(s)

for name in ("TripleGraph", "CoxPH", "GraphSAGE-RNA"):
    if name in PRED_T0:
        rT, cut = PRED_T0[name]
        km, _ = km_groups(rT, tT, eT, cut)
        KM_ROWS.append(dict(model=name, cohort="TCGA (METABRIC-trained, cut = METABRIC training median)", **km))
        lo, hi, _ = bootstrap_c(tT, rT, eT, N_BOOT)
        KM_ROWS[-1].update(c=harrell_c(tT, rT, eT), c_lo=lo, c_hi=hi)
save(pd.DataFrame(KM_ROWS), "km_logrank_hr.csv")

# mutation zero-imputation sensitivity: profiled-only vs all TCGA patients with unprofiled set to wild type
if HAS_TCGA_MUT and (~T.mut_profiled).sum() > 0:
    rows = []
    for seed in SEEDS:
        spM = make_split(eM, seed); prepM, FM = features_S2(spM)
        FA = ext_inputs(prepM, T); tA, eA = T.time.values, T.event.values
        o = run_model(E7_SPECS["TripleGraph"], FM, tM, eM, spM, hp_for("TripleGraph"), seed,
                      externals=[dict(name="all", F=FA, mode="within")])
        prof = T.mut_profiled.values
        rows.append(dict(seed=seed, c_all_zero_imputed=harrell_c(tA, o["ext"]["all"], eA),
                         c_profiled_only=harrell_c(tA[prof], o["ext"]["all"][prof], eA[prof])))
    save(pd.DataFrame(rows), "e7_mutation_imputation_sensitivity.csv")

In [ ]:
# FIGURES
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 10, "savefig.dpi": 300})

def km_plot(ax, rsk, t, e, cut, title):
    g = rsk >= cut
    for lab, i in (("High risk", g), ("Low risk", ~g)):
        KaplanMeierFitter().fit(t[i], e[i], label=f"{lab} (n={i.sum()})").plot_survival_function(ax=ax, ci_show=True)
    km, _ = km_groups(rsk, t, e, cut)
    ax.set_title(f"{title}\nHR {km.get('HR', np.nan):.2f} ({km.get('HR_lo', np.nan):.2f}-{km.get('HR_hi', np.nan):.2f}), "
                 f"log-rank p = {km.get('logrank_p', np.nan):.2g}")
    ax.set_xlabel("Months"); ax.set_ylabel("Survival probability")

fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
rk = PRED0["TripleGraph"]
km_plot(axs[0], rk[te], tM[te], eM[te], np.median(rk[sp0["tr"]]), "METABRIC held-out test (TripleGraph)")
if "TripleGraph" in PRED_T0:
    km_plot(axs[1], PRED_T0["TripleGraph"][0], tT, eT, PRED_T0["TripleGraph"][1], "TCGA external (TripleGraph)")
elif "GraphSAGE-RNA" in PRED_T0:
    km_plot(axs[1], PRED_T0["GraphSAGE-RNA"][0], tT, eT, PRED_T0["GraphSAGE-RNA"][1], "TCGA external (GraphSAGE-RNA)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/figures/fig_km.png"); plt.savefig(f"{OUT_DIR}/figures/fig_km.pdf"); plt.show()

s = pd.read_csv(f"{OUT_DIR}/e2_ablation_summary.csv")
fig, ax = plt.subplots(figsize=(7, 4))
order = ["clin", "mut", "rna", "clin+mut", "clin+rna", "rna+mut", "clin+rna+mut"]
y = np.arange(len(order))
for j, (g, col_) in enumerate((("knn", "#1f77b4"), ("none", "#aaaaaa"))):
    d = s[s.graph == g].set_index("modalities").reindex(order)
    ax.barh(y + (0.2 if j == 0 else -0.2), d["mean"], xerr=d["std"], height=0.38, color=col_,
            label="with graph" if g == "knn" else "no graph (same network)")
ax.set_yticks(y); ax.set_yticklabels(order); ax.set_xlim(0.5, None); ax.axvline(0.5, color="k", lw=0.8)
ax.set_xlabel("Test C-index (mean ± SD over splits)"); ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/figures/fig_ablation.png"); plt.savefig(f"{OUT_DIR}/figures/fig_ablation.pdf"); plt.show()

c = pd.read_csv(f"{OUT_DIR}/e6_calibration_5y.csv"); d = pd.read_csv(f"{OUT_DIR}/e6_decision_curve_5y.csv")
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
for name in ("TripleGraph", "CoxPH", "DeepSurv", "NPI-Cox"):
    cc = c[c.model == name]; axs[0].plot(cc.predicted_S60, cc.observed_KM_S60, "o-", label=name)
    dd = d[d.model == name]; axs[1].plot(dd.threshold, dd.net_benefit_model, label=name)
axs[0].plot([0, 1], [0, 1], "k--", lw=0.8); axs[0].set_xlabel("Predicted 5-year survival"); axs[0].set_ylabel("Observed (KM)")
axs[0].set_title("Calibration, out-of-fold"); axs[0].legend()
dd = d[d.model == "TripleGraph"]
axs[1].plot(dd.threshold, dd.net_benefit_treat_all, "k:", label="treat all"); axs[1].axhline(0, color="k", lw=0.8)
axs[1].set_xlabel("Threshold 5-year risk"); axs[1].set_ylabel("Net benefit"); axs[1].set_title("Decision curve, 5 years"); axs[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/figures/fig_calibration_dca.png"); plt.savefig(f"{OUT_DIR}/figures/fig_calibration_dca.pdf"); plt.show()

In [ ]:
# PACK RESULTS
import platform, sklearn, lifelines, sksurv, torch_geometric
json.dump(dict(python=platform.python_version(), torch=torch.__version__, pyg=torch_geometric.__version__,
               lifelines=lifelines.__version__, sksurv=sksurv.__version__, sklearn=sklearn.__version__,
               device=str(DEVICE)), open(f"{OUT_DIR}/versions.json", "w"), indent=2)
shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print("Share this file:", OUT_DIR + ".zip")

In [ ]:
# MANUSCRIPT FIGURES 3-8 (revised) -> Revision_R1/manuscript_figures/ + manuscript_figures.zip
import os, shutil, numpy as np, pandas as pd, matplotlib.pyplot as plt, networkx as nx
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

BASE = "/content/drive/MyDrive/Thesis"
RUN_OS, RUN_DSS = f"{BASE}/Revision_R1", f"{BASE}/Revision_R1_DSS"
METABRIC_CSV = f"{BASE}/Dataset/METABRIC_RNA_Mutation.csv"
OUT = f"{RUN_OS}/manuscript_figures"
os.makedirs(OUT, exist_ok=True)

BLUE, ORANGE = "#1f77b4", "#ff7f0e"            # colours of the original Figs 4-5
DARK_BLUE, LIGHT_BLUE = "#1565c0", "#9ec5ea"    # colours of the original Fig 6
TAB = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.color": "#e6e6e6", "grid.linewidth": 0.6,
                     "axes.axisbelow": True, "savefig.dpi": 600, "savefig.bbox": "tight"})

def save(name):
    plt.savefig(f"{OUT}/{name}.pdf"); plt.savefig(f"{OUT}/{name}.png"); plt.show(); plt.close()
    print("saved", name)

SUB = ["clin", "mut", "rna", "clin+rna", "clin+mut", "rna+mut", "clin+rna+mut"]
SUBL = {"clin": "Clinical\n(GraphSAGE)", "mut": "Mutation\n(GraphSAGE)", "rna": "RNA\n(GraphSAGE)",
        "clin+rna": "Clinical + RNA\n(DualGraph)", "clin+mut": "Clinical + Mutation\n(DualGraph)",
        "rna+mut": "RNA + Mutation\n(DualGraph)", "clin+rna+mut": "Clinical + RNA + Mutation\n(TripleGraph)"}

# =====================================================================================
# Fig. 3  Modality-specific patient similarity graphs (training patients, split 42, k = 10)
#         nodes coloured by the held-out (out-of-fold) TripleGraph risk score
# =====================================================================================
def fig3():
    met = pd.read_csv(METABRIC_CSV, low_memory=False)
    meta31 = ["patient_id", "age_at_diagnosis", "type_of_breast_surgery", "cancer_type", "cancer_type_detailed",
              "cellularity", "chemotherapy", "pam50_+_claudin-low_subtype", "cohort", "er_status_measured_by_ihc",
              "er_status", "neoplasm_histologic_grade", "her2_status_measured_by_snp6", "her2_status",
              "tumor_other_histologic_subtype", "hormone_therapy", "inferred_menopausal_state", "integrative_cluster",
              "primary_tumor_laterality", "lymph_nodes_examined_positive", "mutation_count",
              "nottingham_prognostic_index", "oncotree_code", "overall_survival_months", "overall_survival",
              "pr_status", "radio_therapy", "3-gene_classifier_subtype", "tumor_size", "tumor_stage", "death_from_cancer"]
    mutc = [c for c in met.columns if c.endswith("_mut")]
    rnac = [c for c in met.columns if c not in meta31 and c not in mutc]
    lm = lambda s, m: s.astype(str).str.strip().str.lower().map(m)
    clin = pd.DataFrame({
        "age": met.age_at_diagnosis,
        "surg": lm(met.type_of_breast_surgery, {"mastectomy": 1, "breast conserving": 0}),
        "cell": lm(met.cellularity, {"low": 0, "moderate": 1, "high": 2}),
        "chemo": met.chemotherapy, "hormone": met.hormone_therapy, "radio": met.radio_therapy,
        "er": lm(met.er_status, {"positive": 1, "negative": 0}), "pr": lm(met.pr_status, {"positive": 1, "negative": 0}),
        "her2": lm(met.her2_status, {"positive": 1, "negative": 0}),
        "meno": lm(met.inferred_menopausal_state, {"post": 1, "pre": 0}),
        "grade": met.neoplasm_histologic_grade, "nodes": met.lymph_nodes_examined_positive,
        "mutcount": met.mutation_count, "npi": met.nottingham_prognostic_index,
        "size": met.tumor_size, "stage": met.tumor_stage}).astype(float)
    rna = met[rnac].astype(float)
    mut = pd.DataFrame({c: ((~met[c].isna()) & (met[c].astype(str).str.strip() != "0")).astype(float) for c in mutc})
    t = met.overall_survival_months.values.astype(float)
    e = (met.overall_survival == 0).astype(int).values            # 1 = died

    idx = np.arange(len(e))                                       # identical to make_split(eM, 42)
    tr, tmp = train_test_split(idx, test_size=0.30, random_state=42, stratify=e)
    tr = np.sort(tr)

    def score(X, tt, ee):                                         # univariate Cox score test (training rows only)
        o = np.argsort(tt, kind="mergesort"); tt, ee, X = tt[o], ee[o], X[o]
        n = len(tt); S0 = np.arange(n, 0, -1, dtype=float)
        S1 = np.cumsum(X[::-1], 0)[::-1]; S2 = np.cumsum((X ** 2)[::-1], 0)[::-1]
        f = np.searchsorted(tt, tt, "left"); S0, S1, S2 = S0[f], S1[f], S2[f]; ev = ee == 1
        m = S1[ev] / S0[ev][:, None]; U = (X[ev] - m).sum(0); I = (S2[ev] / S0[ev][:, None] - m ** 2).sum(0)
        return np.where(I > 1e-12, U ** 2 / np.where(I > 1e-12, I, 1), 0)

    def z(D):
        D = D.iloc[tr].fillna(D.iloc[tr].median())
        return ((D - D.mean()) / D.std(ddof=0).replace(0, 1)).values
    Xc = z(clin)
    Xr_all = z(rna); Xr = Xr_all[:, np.argsort(-score(Xr_all, t[tr], e[tr]))[:100]]
    Mt = mut.iloc[tr]; cand = [c for c in mutc if Mt[c].sum() >= 10]
    Xm = Mt[cand].values[:, np.argsort(-score(Mt[cand].values, t[tr], e[tr]))[:40]]

    oof = pd.read_csv(f"{RUN_OS}/e6_oof_predictions.csv")["risk_TripleGraph"].values[tr]
    er = met.er_status.values[tr]; pam = met["pam50_+_claudin-low_subtype"].replace({"NC": np.nan}).values[tr]

    fig, axs = plt.subplots(1, 3, figsize=(15, 4.6))
    for ax, X, metric, title in zip(axs, (Xc, Xr, Xm), ("euclidean", "cosine", "hamming"),
                                    ("Clinical Patient Similarity Graph", "RNA Expression Similarity Graph",
                                     "Mutation-Based Patient Graph")):
        _, ind = NearestNeighbors(n_neighbors=11, metric=metric, algorithm="brute").fit(X).kneighbors(X)
        edges = {tuple(sorted((a, b))) for a in range(len(X)) for b in [x for x in ind[a] if x != a][:10]}
        same_er = np.mean([er[a] == er[b] for a, b in edges])
        p_er = pd.Series(er).value_counts(normalize=True).values
        ok = [(a, b) for a, b in edges if pd.notna(pam[a]) and pd.notna(pam[b])]
        same_pam = np.mean([pam[a] == pam[b] for a, b in ok])
        p_pam = pd.Series(pam).dropna().value_counts(normalize=True).values
        G = nx.Graph(); G.add_nodes_from(range(len(X))); G.add_edges_from(edges)
        pos = nx.spring_layout(G, k=0.06, iterations=60, seed=0)
        nx.draw_networkx_edges(G, pos, ax=ax, width=0.15, alpha=0.10, edge_color="#7f7f7f")
        sc = nx.draw_networkx_nodes(G, pos, ax=ax, node_size=7, node_color=oof, cmap="coolwarm",
                                    vmin=-2, vmax=2, linewidths=0)
        ax.set_title(f"{title}\nER homophily {same_er:.2f} (random {np.sum(p_er**2):.2f}) · "
                     f"PAM50 {same_pam:.2f} (random {np.sum(p_pam**2):.2f})", fontsize=10)
        ax.axis("off")
        cb = plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.01); cb.set_label("Normalized Risk Score", fontsize=8)
        cb.ax.tick_params(labelsize=7)
    plt.tight_layout(); save("fig3_patient_graphs")

# =====================================================================================
# Fig. 4  Cross-cohort generalisation (same harmonised models: METABRIC test vs TCGA external)
# =====================================================================================
def fig4():
    s = pd.read_csv(f"{RUN_OS}/e7_cross_cohort_summary.csv").query("trained_on == 'METABRIC'")
    order = ["XGBoost-Cox", "CoxPH", "DeepSurv", "TripleGraph-noGraph", "TripleGraph", "DualGraph",
             "UnifiedGraph", "GraphSAGE-RNA"]
    lab = {"XGBoost-Cox": "XGBoost (Cox)", "TripleGraph-noGraph": "TripleGraphSurvival\n(no graph)",
           "TripleGraph": "TripleGraphSurvival", "DualGraph": "DualGraphSurvival", "UnifiedGraph": "Unified graph",
           "GraphSAGE-RNA": "GraphSAGE (RNA)"}
    a = s[s.evaluated_on == "METABRIC test"].set_index("model").reindex(order)
    b = s[(s.evaluated_on == "TCGA all") & (s.graph_mode == "within")].set_index("model").reindex(order)
    y = np.arange(len(order))[::-1]; h = 0.38
    fig, ax = plt.subplots(figsize=(10, 6.2))
    for d, off, col, name in ((a, h / 2, BLUE, "METABRIC (internal test)"), (b, -h / 2, ORANGE, "TCGA (external)")):
        ax.barh(y + off, d.harrell_c_mean - 0.5, left=0.5, height=h, color=col, label=name,
                xerr=d.harrell_c_std, error_kw=dict(ecolor="#404040", elinewidth=0.9, capsize=2))
        for yy, m, sd in zip(y + off, d.harrell_c_mean, d.harrell_c_std):
            ax.text(m + sd + 0.004, yy, f"{m:.3f}", va="center", fontsize=8.5)
    ax.set_yticks(y); ax.set_yticklabels([lab.get(m, m) for m in order], fontsize=9)
    ax.set_xlim(0.5, 0.82); ax.set_xlabel("Concordance Index (C-index), mean ± SD over 10 splits")
    ax.set_title("Cross-Cohort Survival Model Generalization", fontsize=11); ax.grid(axis="y", visible=False)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=2, frameon=False)
    plt.tight_layout(); save("fig4_cross_cohort")

# =====================================================================================
# Fig. 5  Kaplan-Meier (held-out METABRIC test + TCGA), produced by the experiment notebook
# =====================================================================================
def fig5():
    for ext in ("pdf", "png"):
        shutil.copy(f"{RUN_OS}/figures/fig_km.{ext}", f"{OUT}/fig5_km.{ext}")
    print("saved fig5_km (copied from figures/fig_km)")

# =====================================================================================
# Fig. 6  Ablation: every modality subset, with graph vs the identical network without edges
# =====================================================================================
def fig6():
    s = pd.read_csv(f"{RUN_OS}/e2_ablation_summary.csv")
    y = np.arange(len(SUB))[::-1]; h = 0.38
    fig, ax = plt.subplots(figsize=(10, 7))
    for g, off, col, name in (("knn", h / 2, DARK_BLUE, "With patient similarity graph"),
                              ("none", -h / 2, LIGHT_BLUE, "Same network without graph edges")):
        d = s[s.graph == g].set_index("modalities").reindex(SUB)
        ax.barh(y + off, d["mean"] - 0.5, left=0.5, height=h, color=col, label=name,
                xerr=d["std"], error_kw=dict(ecolor="#404040", elinewidth=0.9, capsize=2))
        for yy, m, sd in zip(y + off, d["mean"], d["std"]):
            ax.text(m + sd + 0.003, yy, f"{m:.3f}", va="center", fontsize=9, fontweight="bold")
    ax.set_yticks(y); ax.set_yticklabels([SUBL[m] for m in SUB], fontsize=10, fontweight="bold")
    ax.set_xlim(0.50, 0.75); ax.set_xlabel("Concordance Index (C-index), mean ± SD over 10 splits", fontweight="bold")
    ax.set_title("Ablation Study: Modality Contribution", fontsize=14, fontweight="bold"); ax.grid(axis="y", visible=False)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=2, frameon=False)
    plt.tight_layout(); save("fig6_ablation")

# =====================================================================================
# Fig. 7  5-year calibration and decision curve (out-of-fold, METABRIC)
# =====================================================================================
def fig7():
    c = pd.read_csv(f"{RUN_OS}/e6_calibration_5y.csv"); d = pd.read_csv(f"{RUN_OS}/e6_decision_curve_5y.csv")
    models = ["TripleGraph", "TripleGraph-noGraph", "CoxPH", "DeepSurv", "NPI-Cox"]
    lab = {"TripleGraph": "TripleGraphSurvival", "TripleGraph-noGraph": "TripleGraphSurvival (no graph)", "NPI-Cox": "NPI"}
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.6))
    for m, col in zip(models, TAB):
        cc = c[c.model == m]; axs[0].plot(cc.predicted_S60, cc.observed_KM_S60, "o-", color=col, ms=5, lw=1.6,
                                          label=lab.get(m, m))
        dd = d[d.model == m]; axs[1].plot(dd.threshold, dd.net_benefit_model, color=col, lw=1.8, label=lab.get(m, m))
    lo = min(c.predicted_S60.min(), c.observed_KM_S60.min()) - 0.03
    axs[0].plot([lo, 1], [lo, 1], "k--", lw=0.9, label="Perfect calibration")
    axs[0].set_xlim(lo, 1); axs[0].set_ylim(lo, 1)
    axs[0].set_xlabel("Predicted 5-year survival"); axs[0].set_ylabel("Observed 5-year survival (Kaplan–Meier)")
    axs[0].set_title("(a) Calibration by risk quintile"); axs[0].legend(fontsize=8, loc="upper left")
    dd = d[d.model == models[0]]
    axs[1].plot(dd.threshold, dd.net_benefit_treat_all, color="#7f7f7f", ls=":", lw=1.6, label="Treat all")
    axs[1].axhline(0, color="k", lw=0.9, label="Treat none")
    axs[1].set_ylim(-0.02, max(d.net_benefit_model.max(), dd.net_benefit_treat_all.max()) + 0.02)
    axs[1].set_xlabel("Threshold probability (5-year death)"); axs[1].set_ylabel("Net benefit")
    axs[1].set_title("(b) Decision curve at 5 years"); axs[1].legend(fontsize=8, loc="upper right")
    plt.tight_layout(); save("fig7_calibration_dca")

# =====================================================================================
# Fig. 8  Paired difference (graph - no graph), 95% CI by corrected resampled t-test, OS and DSS
# =====================================================================================
def fig8():
    n_tr, n_te, J = 1332, 286, 10; tq = stats.t.ppf(0.975, J - 1)
    y = np.arange(len(SUB))[::-1]
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    for run, col, off, name in ((RUN_OS, BLUE, 0.14, "Overall survival"), (RUN_DSS, ORANGE, -0.14, "Disease-specific survival")):
        p = pd.read_csv(f"{run}/e2_ablation_paired_tests.csv")
        p = p[p.comparison == "graph - no graph"].set_index("modalities").reindex(SUB)
        ci = tq * np.sqrt(1 / J + n_te / n_tr) * p.sd_diff
        ax.errorbar(p.mean_diff, y + off, xerr=ci, fmt="o", ms=6, color=col, ecolor=col, elinewidth=1.6,
                    capsize=3, label=name)
    ax.axvline(0, color="k", lw=1)
    ax.set_yticks(y); ax.set_yticklabels([SUBL[m].replace("\n", " ") for m in SUB], fontsize=9)
    ax.set_xlabel("Δ C-index (graph − no graph), 95% CI over 10 splits")
    ax.set_title("Effect of the Patient Similarity Graph", fontsize=11)
    ax.text(0.02, 0.02, "← graph worse", transform=ax.transAxes, color="#555555", fontsize=9)
    ax.text(0.98, 0.02, "graph better →", transform=ax.transAxes, color="#555555", fontsize=9, ha="right")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False); ax.grid(axis="y", visible=False)
    plt.tight_layout(); save("fig8_graph_vs_nograph")

for f in (fig3, fig4, fig5, fig6, fig7, fig8):
    try:
        f()
    except Exception as ex:
        print(f"!! {f.__name__} failed: {type(ex).__name__}: {ex}")

shutil.make_archive(OUT, "zip", OUT)
print("\nFiles:", sorted(os.listdir(OUT)))
print("Zip:", OUT + ".zip")